# PFDM 2次元: 直接法 vs CG (スパース性+BTTB+FFT の同時利用)

2次元Poisson方程式に対する $(D_h K D_h^\top + \sigma I)\alpha = b$ を解く：

- **直接法**: $D_h K D_h^\top + \sigma I$ を明示構築してCholesky $O(N^3)$
- **CG + スパース + 2D FFT**:
  1. $w_1 = D_h^\top v$ → スパース積 $O(N)$
  2. $w_2 = K w_1$ → 2次元カーネルは BTTB → 2D FFT で $O(N \log N)$
  3. $w_3 = D_h w_2 + \sigma v$ → スパース積 $O(N)$

$N = Nx \times Ny$，$K$ を**明示的に構築しない**ので $O(N^2)$ のメモリ不要。

In [ ]:
using LinearAlgebra
using SparseArrays
using FFTW
using SpecialFunctions
using IterativeSolvers
using LinearMaps
using Plots
using Statistics
using Printf


In [ ]:
# 2次元 Matérnカーネル
function matern_kernel_2d(x1, x2; ν=1.5, ℓ=1.0, σf=1.0)
    r = sqrt(sum((x1 .- x2).^2))
    r == 0 && return σf^2
    z = sqrt(2ν) * r / ℓ
    if ν == 0.5
        return σf^2 * exp(-z)
    elseif ν == 1.5
        return σf^2 * (1 + z) * exp(-z)
    elseif ν == 2.5
        return σf^2 * (1 + z + z^2/3) * exp(-z)
    else
        return σf^2 * (2^(1-ν)/gamma(ν)) * z^ν * besselk(ν, z)
    end
end

# 2次元 D_h (スパース, 5点ステンシル)
function build_Dh_2d(Nx, Ny, dx)
    N = Nx * Ny
    index(i, j) = i + (j-1)*Nx
    rows, cols, vals = Int[], Int[], Float64[]
    for j in 1:Ny, i in 1:Nx
        k = index(i, j)
        push!(rows, k); push!(cols, k); push!(vals, -4.0/dx^2)
        if i > 1;  push!(rows, k); push!(cols, index(i-1,j)); push!(vals, 1.0/dx^2); end
        if i < Nx; push!(rows, k); push!(cols, index(i+1,j)); push!(vals, 1.0/dx^2); end
        if j > 1;  push!(rows, k); push!(cols, index(i,j-1)); push!(vals, 1.0/dx^2); end
        if j < Ny; push!(rows, k); push!(cols, index(i,j+1)); push!(vals, 1.0/dx^2); end
    end
    return sparse(rows, cols, vals, N, N)
end

# 2次元 BTTB 行列ベクトル積 (2D FFT)
# 定常カーネル k(x-x') on 均一グリッド → BTTB 構造
# 第1列 (Nx×Ny の行列形式) を循環埋め込みで 2D FFT
function bttb_matvec(k_fft2, v, Nx, Ny)
    # v: 長さ Nx*Ny のベクトル → Nx×Ny 行列に reshape
    V = reshape(v, Nx, Ny)
    # ゼロパディング: (2Nx) × (2Ny)
    V_pad = zeros(2Nx, 2Ny)
    V_pad[1:Nx, 1:Ny] .= V
    # 2D FFT で積
    result = real.(ifft2(k_fft2 .* fft2(V_pad)))
    return vec(result[1:Nx, 1:Ny])
end

# BTTB の FFT 埋め込みを事前計算
# k_col: Nx×Ny の第1列行列 (k(x_i - x_1, y_j - y_1))
function bttb_fft_setup(k_col_mat, Nx, Ny)
    # 循環埋め込み: (2Nx) × (2Ny)
    C = zeros(2Nx, 2Ny)
    C[1:Nx, 1:Ny] .= k_col_mat
    # x方向の折り返し
    if Nx > 1
        C[Nx+2:2Nx, 1:Ny] .= k_col_mat[Nx:-1:2, :]
    end
    # y方向の折り返し
    if Ny > 1
        C[1:Nx, Ny+2:2Ny] .= k_col_mat[:, Ny:-1:2]
    end
    # 両方向の折り返し
    if Nx > 1 && Ny > 1
        C[Nx+2:2Nx, Ny+2:2Ny] .= k_col_mat[Nx:-1:2, Ny:-1:2]
    end
    return fft2(C)
end

fft2  = x -> fft(fft(x, 1), 2)
ifft2 = x -> ifft(ifft(x, 1), 2)

u_exact(x, y) = sin(π*x) * sin(π*y) / (2π^2)
f_func(x, y)  = -sin(π*x) * sin(π*y)


In [ ]:
ν   = 1.5
ℓ   = 1.0
σf  = 1.0
C   = 1e-3
τ   = 2

grid_sizes = [5, 8, 10, 15, 20, 30, 50, 70, 100]

results = []

for Nx in grid_sizes
    Ny  = Nx
    N   = Nx * Ny
    dx  = 1.0 / (Nx + 1)

    x = range(dx, step=dx, length=Nx)
    y = range(dx, step=dx, length=Ny)

    X_all  = [[xi, yj] for yj in y for xi in x]
    b_vec  = [f_func(xi, yj)  for yj in y for xi in x]
    u_true = [u_exact(xi, yj) for yj in y for xi in x]

    Dh_sp   = build_Dh_2d(Nx, Ny, dx)
    σ_noise = C * dx^(2τ)

    # BTTB の第1列行列: k(x_i - x_1, y_j - y_1)
    k_col_mat = [matern_kernel_2d([x[i], y[j]], [x[1], y[1]]; ν=ν, ℓ=ℓ, σf=σf)
                 for i in 1:Nx, j in 1:Ny]
    k_fft2 = bttb_fft_setup(k_col_mat, Nx, Ny)

    # ===== 直接法 =====
    t_direct = @elapsed begin
        K     = [matern_kernel_2d(X_all[i], X_all[j]; ν=ν, ℓ=ℓ, σf=σf)
                 for i in 1:N, j in 1:N]
        A     = Dh_sp * K * Dh_sp' + σ_noise * I(N)
        α_dir = cholesky(Hermitian(A)) \ b_vec
        μ_dir = (Dh_sp * K)' * α_dir
    end
    mse_direct = mean((μ_dir .- u_true).^2)

    # ===== CG + スパース + 2D FFT =====
    A_map = LinearMap(
        v -> begin
            w1 = Dh_sp' * v
            w2 = bttb_matvec(k_fft2, w1, Nx, Ny)
            w3 = Dh_sp * w2
            w3 .+ σ_noise .* v
        end,
        N; issymmetric=true, isposdef=true
    )

    t_cg = @elapsed begin
        α_cg, hist = cg(A_map, b_vec; log=true, maxiter=10N, reltol=1e-10)
        μ_cg = bttb_matvec(k_fft2, Dh_sp' * α_cg, Nx, Ny)
    end
    mse_cg = mean((μ_cg .- u_true).^2)
    n_iter = hist.iters

    push!(results, (
        N=N, Nx=Nx,
        t_direct=t_direct, t_cg=t_cg,
        mse_direct=mse_direct, mse_cg=mse_cg,
        n_iter=n_iter
    ))

    @printf("Nx=%3d (N=%5d): direct=%.3gs  CG=%.3gs  iter=%d  MSE_dir=%.3e  MSE_cg=%.3e\n",
        Nx, N, t_direct, t_cg, n_iter, mse_direct, mse_cg)
end


In [ ]:
Ns      = [r.N         for r in results]
t_dir   = [r.t_direct  for r in results]
t_cg_v  = [r.t_cg      for r in results]
mse_dir = [r.mse_direct for r in results]
mse_cg  = [r.mse_cg     for r in results]
n_iters = [r.n_iter     for r in results]

p1 = plot(Ns, t_dir;
    marker=:circle, lw=2, label="Direct: Cholesky",
    xlabel="N = Nx²", ylabel="Time (s)", title="Computation Time (2D, Matérn ν=$ν)",
    yscale=:log10, xscale=:log10, legend=:topleft)
plot!(p1, Ns, t_cg_v;
    marker=:square, lw=2, ls=:dash,
    label="CG (sparse Dh + 2D FFT)")
c3 = t_dir[end] / Ns[end]^3
c1 = t_cg_v[end] / (Ns[end] * log(Ns[end]))
plot!(p1, Ns, c3 .* Ns.^3;          lw=1, ls=:dot, color=:gray,   label="O(N³)")
plot!(p1, Ns, c1 .* Ns .* log.(Ns); lw=1, ls=:dot, color=:orange, label="O(N log N)")

p2 = plot(Ns, mse_dir;
    marker=:circle, lw=2, label="Direct",
    xlabel="N", ylabel="MSE", title="Posterior Mean MSE",
    yscale=:log10, xscale=:log10, legend=:topright)
plot!(p2, Ns, mse_cg;
    marker=:square, lw=2, ls=:dash, label="CG")

p3 = plot(Ns, n_iters;
    marker=:diamond, lw=2, color=:purple,
    xlabel="N", ylabel="CG iterations",
    title="CG Iteration Count",
    xscale=:log10, legend=false)

p_all = plot(p1, p2, p3; layout=(1,3), size=(1200, 400), margin=6Plots.mm)
display(p_all)
savefig(p_all, "pfdm_2d_bttb_cg.png")
println("Saved: pfdm_2d_bttb_cg.png")
